# TikTok research handoff for ChatGPT

This notebook joins the existing project stages without replacing them:

1. `tiktok_search_term_discovery.ipynb` expands a weak seed and preserves first-layer/second-layer lineage.
2. Select one or more useful search terms below.
3. The established collector downloads **N videos per term**, captures public metadata, and runs local Faster-Whisper transcription.
4. This notebook deduplicates repeated video IDs and writes one portable JSON handoff for ChatGPT.
5. It ranks the final corpus by captured views and creates separate top-10 and bottom-10 MP4 folders for direct comparison.

The handoff contains no local-model visual interpretation. ChatGPT receives the native MP4s separately and uses the JSON for search context, metadata and timestamped speech. ASR cannot describe music, sound effects, vocal tone or other non-speech audio.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import re
import subprocess
import sys
import shutil
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
VIDEO_DIR = PROJECT_ROOT / "data" / "videos"
DISCOVERY_FILE = OUTPUT_DIR / "latest_search_terms.json"
COLLECTOR = PROJECT_ROOT / "tiktok_video_search.py"
COLLECTOR_PYTHON = Path(os.environ.get("TIKTOK_ANALYSIS_PYTHON", sys.executable)).expanduser()

# Edit these values for a new run. Terms should be chosen after inspecting the
# nested branches produced by tiktok_search_term_discovery.ipynb.
SELECTED_SEARCH_TERMS = ["creepy tok"]
VIDEOS_PER_TERM = 10
TOP_BOTTOM_N = 10
RUN_COLLECTION = True  # Normal workflow: collect and transcribe a new set.
CLOSE_BROWSER_AFTER_COLLECTION = False
RUN_LABEL = "new_research_run"
HANDOFF_OUTPUT = OUTPUT_DIR / "chatgpt_handoffs" / f"{RUN_LABEL}_handoff.json"
TOP_VIDEO_DIR = VIDEO_DIR / f"{RUN_LABEL}_top_{TOP_BOTTOM_N}_by_views"
BOTTOM_VIDEO_DIR = VIDEO_DIR / f"{RUN_LABEL}_bottom_{TOP_BOTTOM_N}_by_views"

# Used when packaging an existing run. New collection runs are added automatically.
EXISTING_RUNS = []

assert VIDEOS_PER_TERM >= 1
assert TOP_BOTTOM_N >= 1
assert SELECTED_SEARCH_TERMS
print(f"Selected {len(SELECTED_SEARCH_TERMS)} term(s); requesting {VIDEOS_PER_TERM} videos per term.")

## 1. Recover search-term lineage

In [ ]:
if not DISCOVERY_FILE.exists():
    raise FileNotFoundError(
        f"Run tiktok_search_term_discovery.ipynb first; missing {DISCOVERY_FILE}"
    )

discovery = json.loads(DISCOVERY_FILE.read_text(encoding="utf-8"))
records_by_term: dict[str, list[dict[str, Any]]] = {}
for record in discovery.get("records", []):
    records_by_term.setdefault(str(record.get("term", "")).casefold(), []).append(record)

selected_lineage = []
for term in SELECTED_SEARCH_TERMS:
    matches = records_by_term.get(term.casefold(), [])
    if matches:
        for match in matches:
            selected_lineage.append({
                "term": term,
                "selection_source": "discovery_file",
                "order": match.get("order"),
                "parent": match.get("parent"),
                "first_order_term": match.get("first_order_term"),
                "query_path": match.get("query_path"),
            })
    else:
        selected_lineage.append({
            "term": term,
            "selection_source": "manual_selection_not_in_discovery_file",
            "order": None,
            "parent": None,
            "first_order_term": None,
            "query_path": None,
        })

print(f"Seed: {discovery.get('search_term')!r}")
for item in selected_lineage:
    print(f"- {item['term']}: {item['query_path'] or item['selection_source']}")

## 2. Optionally collect and transcribe N videos per term

This invokes the established Version 1 collector with `--local-vision-limit 0`. It downloads media and transcribes speech but performs no local visual-language-model analysis. The isolated Edge session remains open unless `CLOSE_BROWSER_AFTER_COLLECTION` is enabled.

In [ ]:
def safe_slug(value: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", value.casefold()).strip("_")
    return slug or "query"


def collect_term(term: str, limit: int) -> dict[str, Any]:
    slug = safe_slug(term)
    run_dir = OUTPUT_DIR / "chatgpt_handoffs" / RUN_LABEL / "source_runs"
    run_dir.mkdir(parents=True, exist_ok=True)
    metadata_csv = run_dir / f"{slug}_metadata.csv"
    analysis_jsonl = run_dir / f"{slug}_transcript.jsonl"
    command = [
        str(COLLECTOR_PYTHON),
        str(COLLECTOR),
        term,
        "--limit", str(limit),
        "--output", str(metadata_csv),
        "--analyze-videos",
        "--analysis-output", str(analysis_jsonl),
        "--video-dir", str(VIDEO_DIR),
        "--local-vision-limit", "0",
    ]
    if CLOSE_BROWSER_AFTER_COLLECTION:
        command.append("--close-browser")
    environment = os.environ.copy()
    environment["PYTHONIOENCODING"] = "utf-8"
    print(f"Collecting {limit} videos for {term!r}...")
    subprocess.run(command, check=True, cwd=PROJECT_ROOT, env=environment)
    return {
        "search_term": term,
        "metadata_csv": metadata_csv,
        "analysis_jsonl": analysis_jsonl,
    }


source_runs = list(EXISTING_RUNS)
if RUN_COLLECTION:
    source_runs = [collect_term(term, VIDEOS_PER_TERM) for term in SELECTED_SEARCH_TERMS]

if not source_runs:
    raise RuntimeError("No source runs are configured. Enable RUN_COLLECTION or provide EXISTING_RUNS.")
print(f"Packaging {len(source_runs)} source run(s).")

## 3. Build the single ChatGPT handoff file

In [ ]:
def int_or_none(value: Any) -> int | None:
    try:
        if value in (None, ""):
            return None
        return int(float(str(value).replace(",", "")))
    except (TypeError, ValueError):
        return None


def float_or_none(value: Any) -> float | None:
    try:
        if value in (None, ""):
            return None
        return float(value)
    except (TypeError, ValueError):
        return None


def parse_hashtags(value: Any) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value]
    if not value:
        return []
    try:
        parsed = json.loads(str(value))
        if isinstance(parsed, list):
            return [str(item) for item in parsed]
    except json.JSONDecodeError:
        pass
    return re.findall(r"#([\w.]+)", str(value))


def cleaned_transcription(record: dict[str, Any]) -> dict[str, Any]:
    source = record.get("transcription") or {}
    status = str(source.get("transcription_status") or "unavailable")
    quality = str(source.get("quality_status") or "unreviewed")
    raw_text = " ".join(str(source.get("transcript") or "").split())
    segments = source.get("transcript_segments") or []
    usable = status == "ok" and quality != "suspected_hallucination" and bool(raw_text)
    if status == "no_audio":
        effective_status = "no_audio_stream"
    elif quality == "suspected_hallucination":
        effective_status = "suspected_asr_hallucination"
    elif status == "ok" and not raw_text:
        effective_status = "no_speech_detected"
    elif usable:
        effective_status = "usable_transcript"
    else:
        effective_status = status
    clean_segments = []
    if usable:
        for segment in segments:
            text = " ".join(str(segment.get("text") or "").split())
            if text:
                clean_segments.append({
                    "start_seconds": float_or_none(segment.get("start")),
                    "end_seconds": float_or_none(segment.get("end")),
                    "text": text,
                })
    return {
        "status": effective_status,
        "usable_for_language_analysis": usable,
        "language": source.get("language") if usable else None,
        "language_probability": float_or_none(source.get("language_probability")) if usable else None,
        "transcript": raw_text if usable else "",
        "segments": clean_segments,
        "discarded_asr_text": raw_text if raw_text and not usable else "",
        "source_status": status,
        "source_quality_status": quality,
    }


def read_source_run(run: dict[str, Any]) -> list[tuple[dict[str, Any], dict[str, str]]]:
    metadata_path = Path(run["metadata_csv"])
    analysis_path = Path(run["analysis_jsonl"])
    if not metadata_path.exists() or not analysis_path.exists():
        raise FileNotFoundError(f"Incomplete source run: {metadata_path}, {analysis_path}")
    with metadata_path.open(encoding="utf-8-sig", newline="") as handle:
        metadata_by_id = {row["video_id"]: row for row in csv.DictReader(handle)}
    analysis_records = [
        json.loads(line)
        for line in analysis_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    return [(record, metadata_by_id.get(str(record.get("video_id")), {})) for record in analysis_records]


videos_by_id: dict[str, dict[str, Any]] = {}
for run in source_runs:
    configured_term = str(run.get("search_term") or "")
    for record, metadata in read_source_run(run):
        video_id = str(record.get("video_id") or metadata.get("video_id") or "")
        if not video_id:
            raise ValueError("A source record has no video_id.")
        search_term = str(record.get("search_term") or metadata.get("search_term") or configured_term)
        appearance = {
            "search_term": search_term,
            "search_rank": int_or_none(record.get("search_rank") or metadata.get("search_rank")),
        }
        if video_id in videos_by_id:
            if appearance not in videos_by_id[video_id]["search_appearances"]:
                videos_by_id[video_id]["search_appearances"].append(appearance)
            continue
        acquisition = record.get("acquisition") or {}
        technical = record.get("technical_metadata") or {}
        media_path = Path(str(acquisition.get("media_path") or ""))
        engagement = record.get("engagement_metrics") or {}
        videos_by_id[video_id] = {
            "video_id": video_id,
            "media_filename": media_path.name if media_path.name else f"{video_id}.mp4",
            "local_media_path": str(media_path),
            "video_url": record.get("video_url") or metadata.get("video_url"),
            "creator_handle": record.get("creator_handle") or metadata.get("creator_handle"),
            "caption": record.get("caption") or metadata.get("caption") or "",
            "hashtags": parse_hashtags(record.get("hashtags") or metadata.get("hashtags")),
            "published_at_utc": record.get("published_at_utc") or metadata.get("published_at_utc"),
            "collected_at_utc": metadata.get("collected_at"),
            "search_appearances": [appearance],
            "engagement_at_collection": {
                "views": int_or_none(engagement.get("play_count") or metadata.get("play_count") or metadata.get("displayed_view_count")),
                "likes": int_or_none(engagement.get("like_count") or metadata.get("like_count")),
                "comments": int_or_none(engagement.get("comment_count") or metadata.get("comment_count")),
                "shares": int_or_none(engagement.get("share_count") or metadata.get("share_count")),
                "saves": int_or_none(engagement.get("collect_count") or metadata.get("collect_count")),
                "captured_at_utc": engagement.get("captured_at") or metadata.get("collected_at"),
            },
            "music_metadata": {
                "title": metadata.get("music_title") or "",
                "author": metadata.get("music_author") or "",
            },
            "media_validation": {
                "acquisition_status": acquisition.get("acquisition_status"),
                "sha256": acquisition.get("sha256"),
                "file_size_bytes": int_or_none(acquisition.get("file_size_bytes")),
                "duration_seconds": float_or_none(technical.get("decoded_duration_seconds") or acquisition.get("duration_seconds")),
                "width": int_or_none(technical.get("decoded_width") or acquisition.get("width")),
                "height": int_or_none(technical.get("decoded_height") or acquisition.get("height")),
                "fps": float_or_none(technical.get("decoded_fps") or acquisition.get("fps")),
                "video_codec": acquisition.get("vcodec"),
                "audio_codec": acquisition.get("acodec"),
            },
            "transcription": cleaned_transcription(record),
        }

videos = sorted(
    videos_by_id.values(),
    key=lambda item: (
        min((entry.get("search_rank") or 10**9) for entry in item["search_appearances"]),
        item["video_id"],
    ),
)
status_counts = Counter(video["transcription"]["status"] for video in videos)
ranked_by_views = sorted(
    [video for video in videos if video["engagement_at_collection"]["views"] is not None],
    key=lambda item: (item["engagement_at_collection"]["views"], item["video_id"]),
)
if len(ranked_by_views) < TOP_BOTTOM_N:
    raise RuntimeError(
        f"Only {len(ranked_by_views)} videos have view counts; {TOP_BOTTOM_N} are required."
    )
bottom_videos = ranked_by_views[:TOP_BOTTOM_N]
top_videos = list(reversed(ranked_by_views[-TOP_BOTTOM_N:]))
overlap_ids = {video["video_id"] for video in top_videos} & {video["video_id"] for video in bottom_videos}
if overlap_ids:
    print(
        f"Warning: top and bottom groups overlap by {len(overlap_ids)} video(s). "
        f"Collect at least {TOP_BOTTOM_N * 2} unique videos for disjoint groups."
    )

def group_summary(group: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [
        {
            "video_id": video["video_id"],
            "views": video["engagement_at_collection"]["views"],
            "media_filename": video["media_filename"],
        }
        for video in group
    ]

comparison_groups = {
    "ranking_metric": "engagement_at_collection.views",
    "group_size": TOP_BOTTOM_N,
    "videos_with_view_counts": len(ranked_by_views),
    "top_folder": str(TOP_VIDEO_DIR),
    "bottom_folder": str(BOTTOM_VIDEO_DIR),
    "overlap_count": len(overlap_ids),
    "top": group_summary(top_videos),
    "bottom": group_summary(bottom_videos),
}

handoff = {
    "schema_version": "1.0",
    "file_purpose": "Search context, metadata and timestamped speech for ChatGPT analysis of the accompanying TikTok MP4 files.",
    "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "analysis_boundary": {
        "included": ["search lineage", "public TikTok metadata", "download validation", "timestamped ASR"],
        "excluded": ["local visual interpretation", "creative-strategy conclusions", "performance causality"],
        "audio_warning": "Transcription covers detected speech only. Music, sound effects, vocal delivery and other non-speech audio still require separate interpretation.",
    },
    "search_discovery": {
        "source_file": DISCOVERY_FILE.name,
        "seed_term": discovery.get("search_term"),
        "collected_at_utc": discovery.get("collected_at"),
        "selected_terms": selected_lineage,
    },
    "collection": {
        "videos_requested_per_term": VIDEOS_PER_TERM,
        "source_run_count": len(source_runs),
        "records_before_deduplication": sum(len(read_source_run(run)) for run in source_runs),
        "unique_video_count": len(videos),
        "transcription_status_counts": dict(sorted(status_counts.items())),
        "media_filenames": [video["media_filename"] for video in videos],
        "comparison_groups": comparison_groups,
    },
    "videos": videos,
}

HANDOFF_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
temporary_output = HANDOFF_OUTPUT.with_suffix(HANDOFF_OUTPUT.suffix + ".tmp")
temporary_output.write_text(json.dumps(handoff, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
temporary_output.replace(HANDOFF_OUTPUT)

def populate_video_group(folder: Path, group: list[dict[str, Any]]) -> None:
    folder.mkdir(parents=True, exist_ok=True)
    for existing in folder.glob("*.mp4"):
        existing.unlink()
    for video in group:
        source = Path(video["local_media_path"])
        if not source.is_file():
            source = VIDEO_DIR / video["media_filename"]
        if not source.is_file():
            raise FileNotFoundError(f"Missing source video for {video['video_id']}: {source}")
        shutil.copy2(source, folder / video["media_filename"])

populate_video_group(TOP_VIDEO_DIR, top_videos)
populate_video_group(BOTTOM_VIDEO_DIR, bottom_videos)
print(f"Wrote {len(videos)} unique video records to {HANDOFF_OUTPUT}")
print(f"Top {TOP_BOTTOM_N}: {TOP_VIDEO_DIR}")
print(f"Bottom {TOP_BOTTOM_N}: {BOTTOM_VIDEO_DIR}")

## 4. Validate the handoff

In [ ]:
loaded = json.loads(HANDOFF_OUTPUT.read_text(encoding="utf-8"))
loaded_ids = [video["video_id"] for video in loaded["videos"]]
groups = loaded["collection"]["comparison_groups"]
assert len(loaded_ids) == len(set(loaded_ids))
assert loaded["collection"]["unique_video_count"] == len(loaded_ids)
assert all(video["video_url"] for video in loaded["videos"])
assert all(video["media_filename"] for video in loaded["videos"])
assert len(groups["top"]) == TOP_BOTTOM_N
assert len(groups["bottom"]) == TOP_BOTTOM_N
assert groups["top"] == sorted(groups["top"], key=lambda item: item["views"], reverse=True)
assert groups["bottom"] == sorted(groups["bottom"], key=lambda item: item["views"])
for label, folder in (("top", TOP_VIDEO_DIR), ("bottom", BOTTOM_VIDEO_DIR)):
    expected = {item["media_filename"] for item in groups[label]}
    actual = {path.name for path in folder.glob("*.mp4")}
    assert actual == expected, f"{label} folder mismatch: expected {expected}, found {actual}"
assert all(
    not video["transcription"]["transcript"]
    for video in loaded["videos"]
    if not video["transcription"]["usable_for_language_analysis"]
)
assert all(
    segment["start_seconds"] <= segment["end_seconds"] <= video["media_validation"]["duration_seconds"] + 0.1
    for video in loaded["videos"]
    for segment in video["transcription"]["segments"]
)

print("Validation: PASS")
print(json.dumps(loaded["collection"], indent=2, ensure_ascii=False))
print(f"\nTop {TOP_BOTTOM_N} folder: {TOP_VIDEO_DIR}")
print(f"Bottom {TOP_BOTTOM_N} folder: {BOTTOM_VIDEO_DIR}")
print("Send ChatGPT either folder plus this single JSON handoff file.")